# Solución Ejercicio 02b  |  Tema 4 - PRyES
## Estimación por máxima verosimilitud

stats

In [ ]:
# import required packages
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import pandas as pd
from scipy import optimize as opt  # para estimación por máxima verosimilitud

### Se sabe que la variable aleatoria X se distribuye como una normal, pero se desconocen sus parámetros. Se dispone de la muestra de datos almacenada en muestra_15000_norm.csv

In [ ]:
# Leemos los datos de la muestra
x = np.loadtxt('muestra_15000_norm.csv', delimiter=',', skiprows=1)

# tamaño de la muestra
n = len(x)
print(f'La muestra tiene {n} observaciones.')

### a) Representa el histograma de la muestra de datos disponible. ¿Se parece a una normal?

In [ ]:
# Respuesta

nbins = 40 # número de intervalos, ajustar para buena visualización

fig = plt.figure(1, figsize=(14, 4)) # permite indicar el nº de la figura y las dimensiones (ancho y alto)
sns.histplot(data=x, bins=nbins)
sns.rugplot(x, color='r')  # opcional
plt.title(f'Histograma ({nbins} intervalos)')
plt.xlabel('x')
plt.ylabel('Frecuencia absoluta')
plt.show()

print('Sí se parece a una normal.')

### b) Asumiendo que X sigue una distribución normal, estima a partir de la muestra disponible de la v.a. X, y de la mejor manera posible, el valor de los parámetros de la distribución teórica de X. Ten cuidado al elegir el estimador, no debería estar sesgado.

In [ ]:
# Respuesta

print('Podemos utilizar los estimadores media y varianza muestral para estimar la media y la varianza poblacionales de X. '
      'Esto significa calcular directamente la media y la varianza '
      'de la muestra como estimaciones de los parámetros poblacionales.')

# calculamos la media de la muestra como estimación de la media poblacional
mu_est = np.mean(x)

print(f'\nLa media poblacional estimada vale {mu_est}')

# la desviación típica (o la varianza, son equivalentes) de la muestra permite obtener la estimación de la dispersión poblacional

std_est_sesgada = np.std(x) # divide por n
std_est_insesgada = np.std(x, ddof=1) # dividiendo por n-1
print(f'\nLa desviación típica poblacional estimada es {std_est_insesgada}, obtenido a partir del estimador insesgado de la varianza muestral.')
print(f'Si se utiliza el estimador sesgado de la varianza muestral, la desviación típica poblacional estimada sería {std_est_insesgada}, '
      'muy similar a la versión insesgada porque el tamaño muestral es bastante grande.')

### c) Representa en la misma figura el histograma de la muestra de la variable X y la función de densidad de una normal cuyos parámetros sean los estimados en el apartado anterior. Usa la opción stat="density" en la llamada a sns.histplot para mostrar frecuencias relativas en lugar de absolutas, ¿Tiene sentido los valores estimados?

In [ ]:
# Respuesta

# Crea v.a. que representa cómo se distribuye X según nuestra estimación
X = stats.norm(mu_est, std_est_insesgada)

print(f'Creamos la v.a. X~N({mu_est}, std = {std_est_insesgada}) y representamos:')

# representa el histograma junto con la distribución estimada de X
 
xs = np.linspace(X.ppf(0.001)-1, X.ppf(0.999)+1, 1000)


fig = plt.figure(1, figsize=(14, 4)) 

nbins = 'auto'
sns.histplot(data=x, bins=nbins, stat="density")  # <---- densidad, para que sea comparable
sns.rugplot(x, color='r')  # opcional

plt.plot( xs, X.pdf(xs), color="black", linewidth = 4, label = f' N({mu_est}, std = {std_est_insesgada}) estimada')

plt.legend()
plt.title(f'Histograma de la muestra de X y distribución estimada de X')
plt.xlabel('x')
plt.show()

print(f'Si parece que X realmente pueda venir de una normal N({mu_est}, std = {std_est_insesgada}')

### d) Para confirmar el resultado obtenido anteriormente, estima ahora numéricamente por máxima verosimilitud a partir de la muestra disponible los parámetros de la distribución normal que sigue X. Utiliza como punto inicial de búsqueda unos valores razonables, deducidos observando el resultado anterior. ¿Qué estimación es mejor, la anterior o esta por máxima verosimilitud?

In [ ]:
# Respuesta
# define la función de verosimilitud logarítmica para maximizarla

def verosimilitud_log(parameters):
    # sacamos los parámetros
    param1, param2 = parameters
    
    # La función de verosimilitud logarítmica depende del problema y requiere
    # conocer la distribución de la v.a. bajo estudio, que dependerá de los params
    # a estimar por MV

    loc = param1
    escalado = param2
    logLikelihood = np.sum(stats.norm.logpdf(x, loc=loc, scale=escalado))  # <--- logpdf es el log de la pdf, ya está preparado ...
        
    # Como la función minimiza y queremos los params que maximizan la verosimilitud
    # simplemente la multimplicamos por -1: max(loglike) = min(-loglike)
    neg_logLikelihood = -1 * logLikelihood
    return neg_logLikelihood 

In [ ]:
# Maximiza la función de verosimilitud logarítmica (minimizando -1 * funcion)
# usamos como punto inicial de búsqueda
x0 = [99, 9]
mle_model = opt.minimize(verosimilitud_log, np.array(x0), method='L-BFGS-B')

# cogemos los coefs estimados
param1 = mle_model.x[0]
param2 = mle_model.x[1]

print(mle_model)
print('Parámetros estimados por MV:\n'
      f'  - parámetro 1 (media): {param1}\n'
      f'  - parámetro 2 (std):   {param2}')

print(f'\nPor tanto, la normal estimada por máxima verosimilitud es N({param1}, std = {param2})')

print('Es mejor la estimación realizada por el primer método apoyándonos en los estimadores de la media y la varianza muestral '
      'es más preciso, no hay un proceso numérico de optimización. Pero el resultado obtenido por máxima verosimilitud es muy similar.')

### e) Representa en la misma figura el histograma de la muestra de la variable X y las funciones de densidad de las distribuciones estimadas por ambos métodos.¿Se observan diferencias?

In [ ]:
# Respuesta

# Crea v.a. que representa cómo se distribuye X según la estimación por max. vero.
X_max_vero = stats.norm(param1, param2)

print(f'Creamos la v.a. X~N({param1}, std = {param2}) y representamos:')

# representa el histograma junto con la distribuciones estimadas de X
xs = np.linspace(X.ppf(0.001)-1, X.ppf(0.999)+1, 1000)

fig = plt.figure(1, figsize=(14, 4)) 

nbins = 'auto'
sns.histplot(data=x, bins=nbins, stat="density")  # <---- densidad, para que sea comparable
sns.rugplot(x, color='r')  # opcional

plt.plot( xs, X.pdf(xs), color="black", linewidth = 2, label = f' N({mu_est}, std = {std_est_insesgada}) estimada teóricamente')
plt.plot( xs, X_max_vero.pdf(xs), color="green", linewidth = 2, label = f' N({param1}, std = {param2}) estimada numéricamente por MVL')

plt.legend()
plt.title(f'Histograma de la muestra de X y distribuciones estimadas de X')
plt.xlabel('x')
plt.show()

print('Lógicamente, no se observan diferencias visualmente ya que los parámetros son muy similares (aunque no iguales).')